In [ ]:
# Installation des dépendances
!pip install -q \
    pandas \
    requests \
    rapidfuzz \
    openai \
    unidecode

In [ ]:
import os
import re
import json
import math
import unicodedata
import pandas as pd
import requests
from rapidfuzz import fuzz, process as fuzz_process
from openai import OpenAI

# FICHIERS
ANNUAIRE_TXT_FILE   = "annuaire.txt"
ANNUAIRE_CLEAN_FILE = "annuaire_sante_clean.csv"
INPUT_FILE          = "rcp_data_degrade_v2.csv"
CITY_COL            = "Ville"
TEXT_COL            = "Histoire de la maladie"
OUTPUT_FILE         = "resultats_multiagent.csv"

#  SEUILS
FUZZY_CUTOFF      = 72
CONFIDENCE_ACCEPT = 0.80
CONFIDENCE_REVIEW = 0.45

#  GÉOGRAPHIE
CVL_DEPTS_2DIGIT = None

#  OPENAI
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") or ""
    if OPENAI_API_KEY:
        os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

print("Clé OpenAI présente :", bool(OPENAI_API_KEY))

# UTILITAIRES PARTAGÉS

MISSING_VALUES = {
    "", "nan", "none", "null", "na", "n/a",
    "inconnu", "non renseigne", "non renseigné",
    "missing", "vide"
}

def normalize(text: str) -> str:
    """Minuscules + suppression accents + strip."""
    if not text:
        return ""
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text

def normalize_upper(x) -> str:
    """Normalisation majuscule (utilisée pour l'annuaire ANS)."""
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = unicodedata.normalize("NFD", x)
    x = "".join(c for c in x if unicodedata.category(c) != "Mn")
    x = re.sub(r"[^A-Z0-9 ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def is_missing(val) -> bool:
    return normalize(str(val) if val is not None else "") in MISSING_VALUES

def clean_doctor_name(nom: str) -> str:
    """Supprime les préfixes Dr/Docteur résiduels."""
    if not nom:
        return ""
    nom = str(nom).strip()
    nom = re.sub(r"^(Dr\.?|Docteur)\s+", "", nom, flags=re.IGNORECASE).strip()
    return nom

print(" Configuration et utilitaires chargés")

**`RowContext` : mémoire partagée entre agents**



In [ ]:
class RowContext:
    """
    Mémoire partagée pour une ligne du DCC.
    Accumulée par tous les agents ; arbitrée par l'OrchestratorAgent.
    """

    def __init__(self, row_id, ville_initiale):
        self.row_id = row_id
        self.ville_initiale = ville_initiale
        self.hypotheses = []
        self.log = []

        # Ensemble des agents réellement activés pour cette ligne
        self.agents_actives = set()

    def add(
        self,
        scenario: int,
        ville: str,
        score: float,
        source: str,
        type_entite: str = "patient"
    ):

        if is_missing(ville):
            return

        malus = (
            0.15
            if type_entite == "etablissement"
            else 0.0
        )

        score_ajuste = max(
            0.0,
            round(score - malus, 3)
        )

        self.hypotheses.append({
            "scenario": scenario,
            "ville": ville,
            "score": score_ajuste,
            "score_brut": score,
            "source": source,
            "type_entite": type_entite
        })

        self.log.append(
            f"[S{scenario}] '{ville}' "
            f"score={score_ajuste} "
            f"source={source} "
            f"type={type_entite}"
        )

    def best(self):

        if not self.hypotheses:
            return None

        return max(
            self.hypotheses,
            key=lambda h: h["score"]
        )

    def cross_validate(self):

        villes = [
            normalize(h["ville"])
            for h in self.hypotheses
        ]

        for i, h in enumerate(self.hypotheses):
            occ = villes.count(
                normalize(h["ville"])
            )

            if occ > 1:
                self.hypotheses[i]["score"] = min(
                    h["score"] + 0.08 * (occ - 1),
                    1.0
                )

                self.log.append(
                    f"[CrossValidation] "
                    f"'{h['ville']}' confirmé par "
                    f"{occ} agents → score boosté"
                )

    def decision_log(self) -> str:
        return (
            " | ".join(self.log)
            if self.log
            else "aucun_indice"
        )


print("RowContext chargé")

## `ReferentialAgent`


In [ ]:
class ReferentialAgent:

    def __init__(self):
        self.communes_ref = []
        self.noms_norm = []
        self.professionnel_df = pd.DataFrame()
        self._loaded = False

    # Chargement annuaire ANS

    def _build_annuaire(self) -> pd.DataFrame:

        PROFESSIONS_CONNUES = [
            "MEDECIN", "SAGE-FEMME", "CHIRURGIEN-DENTISTE", "PHARMACIEN"
        ]

        with open(ANNUAIRE_TXT_FILE, "r", encoding="utf-8", errors="replace") as f:
            sample_lines = [f.readline() for _ in range(50)]

        sample_lines = [l for l in sample_lines if l.strip()]

        n_cols = max(len(l.rstrip("\n").split("|")) for l in sample_lines)
        print(f"   Colonnes détectées : {n_cols}")

        col_profession_idx = None

        for col_idx in range(3, min(15, n_cols)):
            hits = sum(
                1 for l in sample_lines
                if col_idx < len(l.split("|"))
                and any(
                    p in normalize_upper(l.split("|")[col_idx])
                    for p in PROFESSIONS_CONNUES
                )
            )

            if hits >= len(sample_lines) * 0.3:
                col_profession_idx = col_idx
                break

        if col_profession_idx is None:
            raise ValueError("Colonne profession non détectée dans l'annuaire ANS.")

        col_nom_idx = col_profession_idx - 3
        col_prenom_idx = col_profession_idx - 2
        col_spec_idx = col_profession_idx + 5

        col_names = [f"col_{i}" for i in range(n_cols)]

        raw = pd.read_csv(
            ANNUAIRE_TXT_FILE,
            sep="|",
            dtype=str,
            encoding="utf-8",
            header=None,
            names=col_names,
            on_bad_lines="skip"
        ).fillna("")

        COL_NOM = f"col_{col_nom_idx}"
        COL_PRENOM = f"col_{col_prenom_idx}"
        COL_PROFESSION = f"col_{col_profession_idx}"
        COL_SPEC = f"col_{col_spec_idx}" if col_spec_idx < n_cols else None

        # Garde toutes les professions
        pro = raw.copy()

        if len(pro) == 0:
            print("0 professionnel trouvé. Vérifiez le fichier annuaire.txt.")
            return pd.DataFrame()

        def _find_address(row):
            cp_pat = re.compile(r"^\d{5}$")
            vals = row.tolist()

            for i in range(len(vals) - 1):
                v = str(vals[i]).strip() if vals[i] else ""

                if cp_pat.match(v):
                    for j in range(i + 1, min(i + 4, len(vals))):
                        c = str(vals[j]).strip() if vals[j] else ""

                        if c and not cp_pat.match(c) and not c.isdigit() and len(c) > 2:
                            return v, c

            return "", ""

        addr = pro.apply(_find_address, axis=1)

        pro["code_postal_extrait"] = [a[0] for a in addr]
        pro["commune_extraite"] = [a[1] for a in addr]

        pro = pro[
            (pro["code_postal_extrait"] != "") &
            (pro["commune_extraite"] != "")
        ].copy()

        # On garde tous les départements
        pro["dept_code"] = pro["code_postal_extrait"].str[:2]

        df = pd.DataFrame({
            "nom_professionnel": pro[COL_NOM],
            "prenom_professionnel": pro[COL_PRENOM],
            "nom_complet": (
                pro[COL_PRENOM].fillna("").str.title()
                + " "
                + pro[COL_NOM].fillna("").str.upper()
            ),
            "profession": pro[COL_PROFESSION],
            "specialite": pro[COL_SPEC].fillna("") if COL_SPEC else "",
            "ville_exercice": pro["commune_extraite"],
            "code_postal": pro["code_postal_extrait"],
            "departement": pro["dept_code"],
        })

        df["nom_norm"] = df["nom_professionnel"].apply(normalize_upper).str.lower()
        df["prenom_norm"] = df["prenom_professionnel"].apply(normalize_upper).str.lower()

        df["ville_norm"] = df["ville_exercice"].apply(normalize)

        df = df.drop_duplicates(
            subset=["nom_norm", "prenom_norm", "ville_norm", "code_postal", "profession"]
        )

        df.to_csv(ANNUAIRE_CLEAN_FILE, index=False, encoding="utf-8-sig")

        return df

    # Chargement référentiel communes

    def _load_communes(self) -> list:

        url = "https://geo.api.gouv.fr/communes"

        r = requests.get(
            url,
            params={
                "fields": "nom,codesPostaux,code,departement,region",
                "format": "json"
            },
            timeout=60
        )

        r.raise_for_status()

        result = []

        for c in r.json():
            cps = c.get("codesPostaux", [])
            dept = c.get("departement", {})
            reg = c.get("region", {})
            dept_code = dept.get("code", "")

            result.append({
                "nom": c.get("nom", ""),
                "nom_norm": normalize(c.get("nom", "")),
                "code_postal": cps[0] if cps else "",
                "codes_postaux": cps,
                "code_insee": c.get("code", ""),
                "departement": dept_code,
                "departement_nom": dept.get("nom", ""),
                "region": reg.get("nom", ""),

            })

        return result


    # Point d'entrée


    def load(self, rebuild_annuaire: bool = True):

        print(" [ReferentialAgent] Chargement des ressources…")

        if rebuild_annuaire:
            print("   → Construction annuaire ANS toutes professions, tous départements…")
            self.professionnel_df = self._build_annuaire()
        else:
            self.professionnel_df = pd.read_csv(
                ANNUAIRE_CLEAN_FILE,
                dtype=str,
                encoding="utf-8-sig"
            ).fillna("")

        print(f"    {len(self.professionnel_df)} professionnels dans l'annuaire")

        print("   → Chargement communes nationales…")


        self.communes_ref = self._load_communes()
        self.noms_norm = [c["nom_norm"] for c in self.communes_ref]

        print(f"   {len(self.communes_ref)} communes chargées")

        self._loaded = True

        print(" [ReferentialAgent] Prêt")


    # Services exposés aux autres agents


    def match_city(self, candidate: str, cutoff: int = FUZZY_CUTOFF):

        if is_missing(candidate):
            return None

        cand_norm = normalize(candidate)

        for c in self.communes_ref:
            if cand_norm == c["nom_norm"]:
                return {
                    **c,
                    "type": "exacte",
                    "score": 0.97
                }

        result = fuzz_process.extractOne(
            cand_norm,
            self.noms_norm,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=cutoff
        )

        if result:
            matched_norm, raw_score, _ = result

            for c in self.communes_ref:
                if c["nom_norm"] == matched_norm:
                    return {
                        **c,
                        "type": "orthographe_corrigee",
                        "score": round(raw_score / 100 * 0.90, 3)
                    }

        return None

    def validate_dept_coherence(self, correction: dict, dept_hint) -> dict:

        if not dept_hint or not correction:
            return correction

        c = dict(correction)

        if c.get("departement") == str(dept_hint):
            c["score"] = min(c["score"] + 0.05, 1.0)
            c["coherence_dept"] = True
        else:
            c["coherence_dept"] = False

        return c

    def get_region_status(self, dept_code: str):
        if not dept_code:
            return "Inconnue"

        for c in self.communes_ref:
            if c["departement"] == dept_code:
                return c["region"]

        return "Inconnue"


#  Instanciation globale

referential = ReferentialAgent()

print("ReferentialAgent instancié (pas encore chargé — appeler referential.load())")

#`BaseAgent`

Tous les agents héritent de `BaseAgent` et implémentent `run(ctx, row)`.


In [ ]:
class BaseAgent:

    name: str = "BaseAgent"

    def __init__(self, ref: ReferentialAgent):
        self.ref = ref

    def run(self, ctx: RowContext, row: dict):
        raise NotImplementedError

    def _log(self, ctx: RowContext, msg: str):
        ctx.log.append(f"[{self.name}] {msg}")

print(" BaseAgent défini")


## Bloc 5 — `SpellingAgent` (Scénario 1)

**Cas traité :** la colonne `Ville` contient une valeur présente mais mal orthographiée.  
**Méthode :** matching flou (`rapidfuzz`) contre le référentiel national.


In [ ]:
class SpellingAgent(BaseAgent):

    name = "SpellingAgent"

    def run(self, ctx: RowContext, row: dict):
      # Enregistre que cet agent a été activé
        ctx.agents_actives.add(self.name)
        ville = row.get(CITY_COL, "")

        if is_missing(ville):
            self._log(ctx, "Ville absente — non applicable")
            return

        correction = self.ref.match_city(ville)

        if not correction:
            self._log(ctx, f"'{ville}' non reconnue dans le référentiel national")
            return

        # Bonus cohérence département si disponible
        dept_hint = None
        for col in ["Departement", "dept", "CP", "Code_postal"]:
            v = str(row.get(col, "")).strip()[:2]
            if v.isdigit():
                dept_hint = v
                break
        correction = self.ref.validate_dept_coherence(correction, dept_hint)

        ctx.add(
            scenario=1,
            ville=correction["nom"],
            score=correction["score"],
            source=(
                f"colonne_Ville + API_Geo_nationale "
                f"({correction.get('region_status', 'region_inconnue')})"
            ),
            type_entite="patient"
        )
        self._log(
            ctx,
            f"'{ville}' → '{correction['nom']}' "
            f"| score={correction['score']} "
            f"| région={correction.get('region', 'inconnue')}"
        )

print(" SpellingAgent chargé")


## `TextExtractionAgent`




In [ ]:
class TextExtractionAgent(BaseAgent):

    name = "TextExtractionAgent"

    def _extract_by_llm(self, text: str, api_key: str) -> dict | None:
        if is_missing(text) or not api_key:
            return None

        client = OpenAI(api_key=api_key)

        prompt = f"""
Tu es un assistant d'extraction d'information depuis des comptes rendus médicaux RCP.

Objectif :
Extraire la ville de résidence probable du patient. À défaut, extraire la ville
où le patient a été diagnostiqué ou pris en charge.

Règles, par ordre de priorité :
1. PRIORITÉ 1 — Ville de résidence explicite du patient (ex. "patient demeurant à",
   "résidant à", "domicilié à"). Confidence élevée : 0.8 à 0.95.
2. PRIORITÉ 2 — Si aucune résidence n'est mentionnée : ville de l'établissement où
   le diagnostic a été posé ou le traitement réalisé (hôpital, clinique, centre).
   Confidence plus faible : 0.4 à 0.7, et type_entite = "etablissement".
3. Ne retourne null QUE si aucune ville d'aucune sorte n'est identifiable dans le texte.
4. Si plusieurs villes du même niveau de priorité sont mentionnées, choisis la plus
   probable ; une ville de priorité 1 l'emporte toujours sur une ville de priorité 2.
5. Le champ type_entite doit refléter la nature réelle de la ville retournée :
   "patient" pour une résidence, "etablissement" pour un lieu de soin,
   "medecin" pour le lieu d'exercice d'un médecin cité.
6. Réponds UNIQUEMENT en JSON valide.

Format attendu :
{{
  "ville": null ou "nom de commune",
  "type_entite": "patient|etablissement|medecin|inconnu",
  "confidence": 0.0,
  "justification": "courte explication précisant si c'est la résidence ou le lieu de soin"
}}

Texte RCP :
{text[:6000]}
"""

        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": "Tu extrais des informations géographiques structurées depuis des textes médicaux."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )

            content = response.choices[0].message.content.strip()
            content = content.replace("```json", "").replace("```", "").strip()

            data = json.loads(content)

            data.setdefault("ville", None)
            data.setdefault("type_entite", "inconnu")
            data.setdefault("confidence", 0.0)
            data.setdefault("justification", "")

            data["confidence"] = max(
                0.0,
                min(float(data["confidence"]), 1.0)
            )

            return data

        except Exception as e:
            print("Erreur LLM TextExtractionAgent :", e)
            return None

    def run(self, ctx: RowContext, row: dict, api_key: str = ""):
        ctx.agents_actives.add(self.name)
        texte = row.get(TEXT_COL, "")

        if is_missing(texte):
            self._log(ctx, "Texte absent")
            return

        if not api_key:
            self._log(ctx, "Clé API absente")
            return

        llm = self._extract_by_llm(texte, api_key)

        if not llm:
            self._log(ctx, "Aucun résultat LLM")
            return

        ville = llm.get("ville")
        type_entite = llm.get("type_entite", "inconnu")
        confiance = float(llm.get("confidence", 0.0))
        justification = llm.get("justification", "")

        if is_missing(ville):
            self._log(ctx, f"Aucune ville détectée | {justification}")
            return

        correction = self.ref.match_city(ville)

        if correction:
            score = min(confiance, correction["score"])

            ctx.add(
                scenario=2,
                ville=correction["nom"],
                score=score,
                source="LLM + API_Geo_nationale",
                type_entite=type_entite
            )

            self._log(
                ctx,
                f"'{ville}' → '{correction['nom']}' score={score:.2f} | {justification}"
            )

        else:
            self._log(ctx, f"Ville '{ville}' non reconnue | {justification}")


print("TextExtractionAgent chargé")

## `AddressGeocoderAgent`


In [ ]:
class AddressGeocoderAgent(BaseAgent):
    name = "AddressGeocoderAgent"


    ADDR_PATTERN = re.compile(
    r"\b\d{1,4}\s+(?:rue|avenue|boulevard|bd|place|chemin|route|allée|allee)\s+[A-Za-zÀ-ÖØ-öø-ÿ0-9\s\-']+",
    flags=re.IGNORECASE )

    def _extract_address(self, value: str) -> str:
        if value is None:
            return ""
        m = self.ADDR_PATTERN.search(str(value))
        if m:
            return re.sub(r"\s+", " ", m.group(0)).strip(" .,-;:")
        return ""

    def _geocode(self, address: str) -> dict | None:
        if is_missing(address):
            return None
        try:
            r = requests.get(
                "https://api-adresse.data.gouv.fr/search/",
                params={"q": address, "limit": 10},
                timeout=10
            )
            if r.status_code != 200:
                return None
            features = r.json().get("features", [])
        except Exception as e:
            print(f"Erreur BAN pour '{address}' :", e)
            return None

        best_cand, best_score = None, -1

        for f in features:
            props   = f.get("properties", {})
            ville_b = props.get("city", "")
            cp      = props.get("postcode", "")
            label   = props.get("label", "")
            score_b = props.get("score", 0)

            if is_missing(ville_b):
                continue

            dept_code     = str(cp)[:2] if cp else ""
            region_status = self.ref.get_region_status(dept_code)

            # Validation dans le référentiel
            corr = self.ref.match_city(ville_b)
            if corr:
                ville_f       = corr["nom"]
                dept_code     = corr.get("departement", dept_code)
                region_status = corr.get("region", region_status)
            else:
                ville_f = ville_b

            if score_b > best_score:
                best_score = score_b
                best_cand  = {
                    "ville": ville_f,
                    "code_postal": cp,
                    "departement": dept_code,
                    "region_status": region_status,
                    "score_ban": score_b,
                    "adresse_ban": label,
                }

        return best_cand

    def run(self, ctx: RowContext, row: dict):
        ctx.agents_actives.add(self.name)
        ville   = row.get(CITY_COL, "")
        adresse = self._extract_address(ville)

        if not adresse:
            self._log(ctx, "Aucune adresse détectée dans la colonne Ville")
            return

        result = self._geocode(adresse)
        if result:
            score_final = round(result["score_ban"] * 0.90, 3)
            ctx.add(
                scenario=3,
                ville=result["ville"],
                score=score_final,
                source=(
                    f"Adresse_BAN ({result['adresse_ban']}) "
                    f"+ API_BAN_nationale ({result['region_status']})"
                ),
                type_entite="patient"
            )
            self._log(
                ctx,
                f"Adresse '{adresse}' → {result['ville']} "
                f"| score_BAN={result['score_ban']} | score_final={score_final} "
                f"| {result['region_status']}"
            )
        else:
            self._log(ctx, f"Adresse '{adresse}' non géocodée par l'API BAN")

print(" AddressGeocoderAgent chargé")


## `PostalCodeAgent`


In [ ]:
class PostalCodeAgent(BaseAgent):
    name = "PostalCodeAgent"

    LLM_CP_CONFIDENCE_MIN = 0.70

    @staticmethod
    def extract_valid_cp(value) -> str:
        if value is None:
            return ""
        value = re.sub(r"\.0$", "", str(value).strip())
        m = re.search(r"\b\d{5}\b", value)
        return m.group(0) if m else ""

    @staticmethod
    def is_invalid_cp_like(value) -> bool:
        if value is None:
            return False
        value = re.sub(r"\.0$", "", str(value).strip())
        if re.fullmatch(r"\d{5}", value):
            return False
        if re.fullmatch(r"\d{1,4}", value):
            return True
        if re.search(r"\b\d{1,4}\b", value) and not re.search(r"\b\d{5}\b", value):
            return True
        return False

    @staticmethod
    def city_name_from_cp_cell(value: str) -> str:
        cleaned = re.sub(r"\b\d{5}\b", "", str(value)).strip(" .,-;:")
        if cleaned and re.match(r"[A-ZÉÈÀÂÊÎÔÛÇA-Za-z]", cleaned):
            return cleaned.strip()
        return ""

    def _resolve_cp(self, cp: str) -> list:
        if not re.fullmatch(r"\d{5}", cp):
            return []
        try:
            r = requests.get(
                "https://geo.api.gouv.fr/communes",
                params={"codePostal": cp,
                        "fields": "nom,code,codesPostaux,departement,region",
                        "format": "json"},
                timeout=10
            )
            if r.status_code != 200:
                return []
            results = []
            for c in r.json():
                dept_code     = c.get("departement", {}).get("code", "")
                region_name   = c.get("region", {}).get("nom", "")
                region_status = self.ref.get_region_status(dept_code)
                results.append({
                    "nom":             c.get("nom", ""),
                    "nom_norm":        normalize(c.get("nom", "")),
                    "code_postal":     cp,
                    "departement":     dept_code,
                    "departement_nom": c.get("departement", {}).get("nom", ""),
                    "region":          region_name,
                    "region_status":   region_status,
                })
            return results
        except Exception as e:
            print(f"Erreur API Géo pour CP {cp} :", e)
            return []

    def _disambiguate_llm(self, text: str, cp: str,
                          candidates: list, api_key: str) -> dict | None:
        if not api_key or is_missing(text) or not candidates:
            return None
        client = OpenAI(api_key=api_key)
        cand_list = [
            {"ville": c["nom"],
             "departement": c.get("departement_nom", ""),
             "region": c.get("region", ""),
             "region_status": c.get("region_status", "")}
            for c in candidates
        ]
        prompt = f"""Choisis la commune la plus probable parmi :
{json.dumps(cand_list, ensure_ascii=False, indent=2)}

Code postal : {cp}
Contexte RCP : {str(text)[:4000]}

Règles : priorise la résidence du patient. Si incertain → ville = null.
Réponds UNIQUEMENT en JSON :
{{"ville": null ou "nom exact", "confidence": 0.0, "justification": ""}}"""
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system",
                     "content": "Tu désambiguïses une commune depuis un contexte médical."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0
            )
            content = response.choices[0].message.content.strip()
            content = content.replace("```json", "").replace("```", "").strip()
            data = json.loads(content)
            data.setdefault("ville", None)
            data.setdefault("confidence", 0.0)
            data.setdefault("justification", "")
            data["confidence"] = max(0.0, min(float(data["confidence"]), 1.0))
            return data
        except Exception as e:
            print("Erreur LLM PostalCodeAgent :", e)
            return None

    def run(self, ctx: RowContext, row: dict, api_key: str = "",
            text_extraction_agent=None):
        ctx.agents_actives.add(self.name)
        ville = row.get(CITY_COL, "")
        cp    = self.extract_valid_cp(ville)

        #  Cas 1 : CP valide
        if cp:
            nom_cellule = self.city_name_from_cp_cell(ville)
            communes    = self._resolve_cp(cp)

            if not communes:
                self._log(ctx, f"CP {cp} valide mais inconnu de l'API Géo")
                return

            # Une seule commune
            if len(communes) == 1:
                c = communes[0]
                ctx.add(
                    scenario=5, ville=c["nom"], score=0.90,
                    source=f"CP_valide + API_Geo_nationale ({c['region_status']})",
                    type_entite="patient"
                )
                self._log(ctx, f"CP {cp} → {c['nom']} | {c['region_status']}")
                return

            # Plusieurs communes
            resolved = False

            # B1 : nom présent dans la cellule
            if nom_cellule:
                nc_norm = normalize(nom_cellule)
                for c in communes:
                    if c["nom_norm"] == nc_norm:
                        ctx.add(
                            scenario=5, ville=c["nom"], score=0.80,
                            source=(
                                f"CP_valide + nom_cellule + API_Geo_nationale "
                                f"({c['region_status']})"
                            ),
                            type_entite="patient"
                        )
                        self._log(ctx, f"CP {cp} ambigu → résolu par nom cellule : {c['nom']}")
                        resolved = True
                        break

            # B2 : désambiguïsation LLM
            if not resolved:
                texte_patient = row.get(TEXT_COL, "")
                llm = self._disambiguate_llm(texte_patient, cp, communes, api_key)
                if llm and not is_missing(llm.get("ville")):
                    conf = float(llm.get("confidence", 0.0))
                    just = llm.get("justification", "")
                    if conf >= self.LLM_CP_CONFIDENCE_MIN:
                        vc = normalize(llm["ville"])
                        for c in communes:
                            if c["nom_norm"] == vc:
                                ctx.add(
                                    scenario=5, ville=c["nom"],
                                    score=min(0.80, conf),
                                    source=(
                                        f"CP_ambigu + LLM + API_Geo_nationale "
                                        f"({c['region_status']})"
                                    ),
                                    type_entite="patient"
                                )
                                self._log(ctx, f"CP {cp} → LLM : {c['nom']} conf={conf} | {just}")
                                resolved = True
                                break
                    else:
                        self._log(ctx, f"CP {cp} LLM pas assez sûr (conf={conf}) | {just}")
                else:
                    just = llm.get("justification", "") if llm else ""
                    self._log(ctx, f"CP {cp} ambigu : LLM n'a pas choisi | {just}")

            if not resolved:
                self._log(ctx, f"CP {cp} ambigu non résolu | candidates={[c['nom'] for c in communes]}")
            return

        #  Cas 2 : CP invalide / incomplet
        if self.is_invalid_cp_like(ville):
            self._log(ctx, f"CP invalide '{ville}' → délégation au TextExtractionAgent")
            if text_extraction_agent:
                text_extraction_agent.run(ctx, row, api_key=api_key)
            return

        #  Cas 3 : pas un CP
        self._log(ctx, f"'{ville}' n'est pas un code postal")

print(" PostalCodeAgent chargé")


## `DoctorLookupAgent`



In [ ]:
class ProfessionalLookupAgent(BaseAgent):

    name = "ProfessionalLookupAgent"

    LLM_PROMPT = """Tu analyses le texte d'une fiche médicale RCP en oncologie.

Objectif :
Extraire les noms de professionnels de santé RÉELS et COMPLETS explicitement mentionnés.

Professionnels possibles :
médecin, chirurgien, oncologue, radiothérapeute, pharmacien, sage-femme,
chirurgien-dentiste, infirmier, kinésithérapeute, etc.

Règles STRICTES :
1. "nom" = uniquement le nom de famille, SANS "Dr", "Docteur", "Mme", "M.".
2. "prenom" = prénom si disponible, sinon chaîne vide.
3. IGNORE toutes les formes anonymisées : "Dr X", "Docteur X", "Mme X", etc.
4. N'invente aucun nom.
5. Si aucun professionnel réel → professionnels = [].
6. Réponds UNIQUEMENT en JSON valide.

Format :
{{
  "professionnels": [
    {{
      "nom": "NOM_DE_FAMILLE",
      "prenom": "PRENOM ou vide",
      "profession": "profession ou vide",
      "specialite": "spécialité ou vide",
      "etablissement": "établissement ou vide"
    }}
  ],
  "justification": "explication courte"
}}

Texte médical :
\"\"\"{texte}\"\"\"
"""

    def _clean_professional_name(self, name: str) -> str:
        if is_missing(name):
            return ""

        name = str(name).strip()
        name = re.sub(
            r"\b(Dr|Docteur|Mme|Madame|M\.|Monsieur)\b\.?",
            "",
            name,
            flags=re.IGNORECASE
        )
        name = re.sub(r"\s+", " ", name).strip(" .,-;:")

        return name

    def _extract_professionals(self, texte: str, api_key: str) -> list:
        if is_missing(texte) or not api_key:
            return []

        client = OpenAI(api_key=api_key)
        prompt = self.LLM_PROMPT.format(texte=str(texte)[:4000])

        try:
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                temperature=0,
                max_tokens=700,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Tu extrais des noms de professionnels de santé "
                            "depuis des textes médicaux. JSON uniquement."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            content = resp.choices[0].message.content.strip()
            content = content.replace("```json", "").replace("```", "").strip()

            data = json.loads(content)
            professionnels = data.get("professionnels", [])

            cleaned = []

            for p in professionnels:
                nom = self._clean_professional_name(p.get("nom", ""))
                prenom = self._clean_professional_name(p.get("prenom", ""))

                if nom.strip() in ("", "X", "x"):
                    continue

                if len(nom.strip()) <= 1:
                    continue

                cleaned.append({
                    "nom": nom,
                    "prenom": prenom,
                    "profession": p.get("profession", ""),
                    "specialite": p.get("specialite", ""),
                    "etablissement": p.get("etablissement", "")
                })

            return cleaned

        except Exception as e:
            print("Erreur LLM ProfessionalLookupAgent :", e)
            return []

    def _search_annuaire(self, nom: str, prenom: str = "") -> dict | None:
        nom = self._clean_professional_name(nom)
        prenom = self._clean_professional_name(prenom)

        if is_missing(nom) or nom.strip() in ("", "X", "x"):
            return None

        df_a = self.ref.professionnel_df.copy()

        if df_a.empty:
            return None

        nom_norm = normalize(nom)
        prenom_norm = normalize(prenom)

        def _pick(subset, conf):
            if len(subset) >= 1:
                r = subset.iloc[0]

                return {
                    "ville": r.get("ville_exercice", ""),
                    "cp": r.get("code_postal", ""),
                    "nom_ps": r.get("nom_complet", ""),
                    "profession": r.get("profession", ""),
                    "specialite": r.get("specialite", ""),
                    "confidence": conf
                }

            return None

        same_name = df_a[df_a["nom_norm"] == nom_norm]

        # Nom + prénom exact
        if prenom_norm:
            exact = same_name[same_name["prenom_norm"] == prenom_norm]

            if len(exact) >= 1:
                return _pick(exact, 0.70 if len(exact) == 1 else 0.55)

        # Nom seul
        if len(same_name) >= 1 and not prenom_norm:
            return _pick(same_name, 0.50)

        # Nom trouvé, mais prénom différent ou absent dans l'annuaire
        if len(same_name) >= 1 and prenom_norm:
            return _pick(same_name, 0.45)

        # Cas où le LLM a mis prénom + nom ensemble
        words = nom_norm.split()

        if len(words) >= 2:

            def try_match(nom_test, prenom_test, conf_exact, conf_nom):
                cand = df_a[df_a["nom_norm"] == nom_test]

                if len(cand) == 0:
                    return None

                if prenom_test:
                    with_p = cand[cand["prenom_norm"] == prenom_test]

                    if len(with_p) >= 1:
                        return _pick(with_p, conf_exact)

                return _pick(cand, conf_nom)

            result = try_match(
                words[-1],
                " ".join(words[:-1]),
                0.55,
                0.45
            )

            if result:
                return result

            if len(words) >= 3:
                result = try_match(
                    " ".join(words[-2:]),
                    words[0],
                    0.55,
                    0.45
                )

                if result:
                    return result

            result = try_match(
                words[0],
                " ".join(words[1:]),
                0.55,
                0.45
            )

            if result:
                return result

        return None

    def run(self, ctx: RowContext, row: dict, api_key: str = ""):
        if not api_key:
            self._log(ctx, "Désactivé — clé OpenAI manquante")
            return

        texte = row.get(TEXT_COL, "")

        if is_missing(texte):
            self._log(ctx, "Texte absent")
            return

        professionnels = self._extract_professionals(texte, api_key)

        if not professionnels:
            self._log(ctx, "Aucun professionnel réel identifié par LLM")
            return

        self._log(
            ctx,
            f"{len(professionnels)} professionnel(s) identifié(s) : "
            f"{[p.get('nom') for p in professionnels]}"
        )

        for pro in professionnels[:3]:
            nom = self._clean_professional_name(pro.get("nom", ""))
            prenom = self._clean_professional_name(pro.get("prenom", ""))

            if not nom or nom.upper() == "X":
                continue

            res = self._search_annuaire(nom, prenom)

            if not res:
                self._log(
                    ctx,
                    f"{prenom} {nom} non trouvé dans l'annuaire"
                )
                continue

            ville_ann = res.get("ville", "")

            if is_missing(ville_ann):
                self._log(
                    ctx,
                    f"{prenom} {nom} trouvé mais ville d'exercice absente"
                )
                continue

            corr = self.ref.match_city(ville_ann)

            if corr:
                score_final = min(res.get("confidence", 0.50), 0.55)

                ctx.add(
                    scenario=4,
                    ville=corr["nom"],
                    score=score_final,
                    source=(
                        f"Histoire_de_la_maladie + LLM + Annuaire_santé "
                        f"({res.get('nom_ps', nom)} - {res.get('profession', '')}) "
                        f"+ API_Geo_nationale "
                        f"({corr.get('region', 'region_inconnue')})"
                    ),
                    type_entite="professionnel"
                )

                self._log(
                    ctx,
                    f"{prenom} {nom} → {corr['nom']} "
                    f"| profession={res.get('profession', '')} "
                    f"| score={score_final} "
                    f"| {corr.get('region', 'region_inconnue')}"
                )

            else:
                self._log(
                    ctx,
                    f"Ville '{ville_ann}' du professionnel {nom} non reconnue"
                )


print("ProfessionalLookupAgent chargé")

## `OrchestratorAgent`


In [ ]:
import time

class OrchestratorAgent:

    def __init__(
        self,
        ref: ReferentialAgent,
        spelling_agent: SpellingAgent,
        text_agent: TextExtractionAgent,
        geocoder_agent: AddressGeocoderAgent,
        postal_agent: PostalCodeAgent,
        professional_agent: ProfessionalLookupAgent,
        api_key: str = "",
        cout_par_appel: float = 0.00015,
    ):
        self.ref = ref
        self.spelling = spelling_agent
        self.text = text_agent
        self.geocoder = geocoder_agent
        self.postal = postal_agent
        self.professional = professional_agent
        self.api_key = api_key
        self.cout_par_appel = cout_par_appel

    def _detect_type(self, row: dict) -> str:
        ville = row.get(CITY_COL, "")
        ville_str = str(ville).strip() if not is_missing(ville) else ""

        if is_missing(ville):
            return "absente"

        if self.geocoder._extract_address(ville_str):
            return "adresse"

        cp = PostalCodeAgent.extract_valid_cp(ville_str)
        if cp:
            return "code_postal"

        if PostalCodeAgent.is_invalid_cp_like(ville_str):
            return "code_postal_invalide"

        ville_norm = normalize(ville_str)

        for c in self.ref.communes_ref:
            if c["nom_norm"] == ville_norm:
                return "saine"

        return "ortho"

    def _count_llm_calls(self, before_log, after_log):
        nouveaux_logs = after_log[len(before_log):]
        count = 0

        for log in nouveaux_logs:
            if "LLM" in log or "llm" in log:
                count += 1

        return count

    def process_row(self, idx: int, row: dict) -> dict:
        t0 = time.perf_counter()

        ville_type = self._detect_type(row)
        ctx = RowContext(row_id=idx, ville_initiale=row.get(CITY_COL, ""))

        n_appels_llm = 0

        if ville_type == "saine":
            best = {
                "ville": row.get(CITY_COL, ""),
                "score": 1.0,
                "source": "inchangee",
                "scenario": 0,
                "type_entite": "patient"
            }
            ville_statut = "saine"

        else:
            if ville_type == "ortho":
                self.spelling.run(ctx, row)

            elif ville_type == "absente":
                before = list(ctx.log)
                self.text.run(ctx, row, api_key=self.api_key)
                n_appels_llm += self._count_llm_calls(before, ctx.log)

                if not ctx.hypotheses:
                    ctx.log.append(
                        "[Orchestrateur] TextExtractionAgent sans résultat → ProfessionalLookupAgent"
                    )

                    before = list(ctx.log)
                    self.professional.run(ctx, row, api_key=self.api_key)
                    n_appels_llm += self._count_llm_calls(before, ctx.log)

            elif ville_type == "adresse":
                self.geocoder.run(ctx, row)

            elif ville_type == "code_postal":
                before = list(ctx.log)
                self.postal.run(
                    ctx,
                    row,
                    api_key=self.api_key,
                    text_extraction_agent=self.text
                )
                n_appels_llm += self._count_llm_calls(before, ctx.log)

            elif ville_type == "code_postal_invalide":
                before = list(ctx.log)
                self.postal.run(
                    ctx,
                    row,
                    api_key=self.api_key,
                    text_extraction_agent=self.text
                )
                n_appels_llm += self._count_llm_calls(before, ctx.log)

                if not ctx.hypotheses:
                    ctx.log.append(
                        "[Orchestrateur] PostalCodeAgent sans résultat → ProfessionalLookupAgent"
                    )

                    before = list(ctx.log)
                    self.professional.run(ctx, row, api_key=self.api_key)
                    n_appels_llm += self._count_llm_calls(before, ctx.log)

            ctx.cross_validate()
            best = ctx.best()

            if best is None:
                ville_statut = "non_resolu"
                best = {
                    "ville": "",
                    "score": 0.0,
                    "source": "aucune",
                    "scenario": None,
                    "type_entite": "inconnu"
                }

            elif best["score"] >= CONFIDENCE_ACCEPT:
                ville_statut = "corrige_auto"

            elif best["score"] >= CONFIDENCE_REVIEW:
                ville_statut = "a_verifier"

            else:
                ville_statut = "non_resolu"

        latence_s = round(time.perf_counter() - t0, 3)

        source = best.get("source", "")
        agent_gagnant = "inconnu"

        if best.get("scenario") == 0:
            agent_gagnant = "Aucun"
        elif best.get("scenario") == 1:
            agent_gagnant = "SpellingAgent"
        elif best.get("scenario") == 2:
            agent_gagnant = "TextExtractionAgent"
        elif best.get("scenario") == 3:
            agent_gagnant = "AddressGeocoderAgent"
        elif best.get("scenario") == 4:
            agent_gagnant = "ProfessionalLookupAgent"
        elif best.get("scenario") == 5:
            agent_gagnant = "PostalCodeAgent"

        result = row.copy()

        result["ville_type_degradation"] = ville_type
        result["ville_enrichie"] = best["ville"]
        result["ville_statut"] = ville_statut
        result["ville_score"] = best["score"]
        result["ville_source"] = source
        result["ville_scenario"] = best["scenario"]
        result["ville_type_entite"] = best["type_entite"]
        result["ville_decision_log"] = ctx.decision_log()

        result["agent_gagnant"] = agent_gagnant
        result["nb_hypotheses"] = len(ctx.hypotheses)
        result["cross_validation"] = any(
            "CrossValidation" in log for log in ctx.log
        )

        result["n_appels_llm"] = n_appels_llm
        result["latence_s"] = latence_s
        result["cout_estime"] = n_appels_llm * self.cout_par_appel
        result["agents_actives"] = ", ".join(sorted(ctx.agents_actives))

        result["nb_agents_actives"] = len(ctx.agents_actives)

        return result

    def run_pipeline(self, df: pd.DataFrame) -> pd.DataFrame:
        resultats = []
        n = len(df)

        for idx, row in df.iterrows():
            if idx % 50 == 0:
                print(f"   … {idx}/{n} lignes traitées")

            resultats.append(
                self.process_row(idx, row.to_dict())
            )

        df_result = pd.DataFrame(resultats)

        print(f"Pipeline terminé — {n} lignes traitées")

        return df_result


print("OrchestratorAgent chargé")

In [ ]:
def load_csv(path: str) -> pd.DataFrame:
    attempts = [
        {"sep": ";",  "encoding": "utf-8-sig"},
        {"sep": ";",  "encoding": "latin1"},
        {"sep": ";",  "encoding": "cp1252"},
        {"sep": ",",  "encoding": "utf-8-sig"},
        {"sep": ",",  "encoding": "latin1"},
        {"sep": None, "encoding": "utf-8-sig"},
    ]
    for a in attempts:
        try:
            df = pd.read_csv(
                path, sep=a["sep"], encoding=a["encoding"],
                engine="python", on_bad_lines="skip"
            )
            df.columns = [c.strip() for c in df.columns]
            if CITY_COL in df.columns and TEXT_COL in df.columns:
                print(f" {path} chargé : {df.shape[0]} lignes | sep={a['sep']} | enc={a['encoding']}")
                return df
        except Exception:
            continue
    raise ValueError(f"Impossible de charger {path}")

df = load_csv(INPUT_FILE)
df = df.reset_index(drop=True)
df["fiche_id_auto"] = df.index + 1
print(f" fiche_id_auto créé — {len(df)} lignes")


## Initialisation et chargement de tous les agents


In [ ]:
# Chargement du référentiel
referential.load(rebuild_annuaire=True)

#  Instanciation des agents spécialisés
spelling_agent = SpellingAgent(referential)
text_agent = TextExtractionAgent(referential)
geocoder_agent = AddressGeocoderAgent(referential)
postal_agent = PostalCodeAgent(referential)
professional_agent = ProfessionalLookupAgent(referential)

#Orchestrateur
orchestrator = OrchestratorAgent(
    ref=referential,
    spelling_agent=spelling_agent,
    text_agent=text_agent,
    geocoder_agent=geocoder_agent,
    postal_agent=postal_agent,
    professional_agent=professional_agent,
    api_key=OPENAI_API_KEY,
    cout_par_appel=0.00015
)

#  Export
print("Tous les agents sont prêts")

## Exécution du pipeline multi-agent

In [ ]:
print(" Lancement du pipeline multi-agent…")
df_result = orchestrator.run_pipeline(df)

## Evaluation

In [ ]:
# NORMALISATION

def normalize_eval(value):
    """
    Normalise les noms de communes pour permettre une comparaison
    robuste entre la prédiction et la vérité terrain.
    """
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    text = re.sub(r"[^a-z0-9\s\-']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_label(value):
    """
    Normalise les libellés catégoriels tels que :
    'Saine', ' saine ', 'SAINE' -> 'saine'
    """
    if value is None or pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    text = re.sub(r"\s+", " ", text).strip()

    return text


def _fix_mojibake(value):
    """
    Corrige certains problèmes d'encodage :
    vÃ©ritÃ© -> vérité
    """
    if not isinstance(value, str):
        return value

    try:
        return value.encode("latin1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return value


def _is_missing_eval(value):
    """Indique si une valeur est absente."""
    if value is None or pd.isna(value):
        return True

    return str(value).strip().lower() in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "n/a"
    }


# ÉVALUATION DU MULTI-AGENT

def evaluate_mas_with_city_ground_truth(
    df_result,
    ground_truth_path="rcp_data_degrade_v2_solution.csv",
    architecture="Multi-Agent",
    validator=None,
    export_path=None,
    healthy_labels=("saine",),
    debug=True
):

    out = df_result.copy().reset_index(drop=True)

    # VÉRIFICATION DES COLONNES

    required_columns = {
        "Ville",
        "ville_enrichie",
        "ville_statut",
        "ville_type_degradation"
    }

    missing_columns = required_columns.difference(out.columns)

    if missing_columns:
        raise ValueError(
            "Colonnes manquantes dans le DataFrame MAS : "
            f"{sorted(missing_columns)}"
        )

    # LECTURE DE LA VÉRITÉ TERRAIN

    try:
        gt = pd.read_csv(
            ground_truth_path,
            sep=";",
            dtype=str,
            keep_default_na=False,
            skip_blank_lines=False,
            encoding="utf-8-sig"
        ).reset_index(drop=True)

    except UnicodeDecodeError:
        gt = pd.read_csv(
            ground_truth_path,
            sep=";",
            dtype=str,
            keep_default_na=False,
            skip_blank_lines=False,
            encoding="latin1"
        ).reset_index(drop=True)

    gt.columns = [
        _fix_mojibake(column.strip())
        for column in gt.columns
    ]

    for column in gt.columns:
        gt[column] = gt[column].map(_fix_mojibake)

    print("Résultats MAS :", out.shape)
    print("Ground truth :", gt.shape)

    if "Ville vérité" not in gt.columns:
        raise ValueError(
            "La colonne 'Ville vérité' est absente. "
            f"Colonnes disponibles : {gt.columns.tolist()}"
        )

    if len(out) != len(gt):
        raise ValueError(
            "Nombre de lignes différent entre les résultats et la vérité terrain : "
            f"{len(out)} contre {len(gt)}"
        )

    # ALIGNEMENT AVEC LA VÉRITÉ TERRAIN

    out["ville_reference"] = (
        gt["Ville vérité"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    out["ground_truth_disponible"] = out["ville_reference"].ne("")
    annotated_mask = out["ground_truth_disponible"]

    print(
        f"Lignes annotées : "
        f"{int(annotated_mask.sum())} / {len(out)}"
    )

    out["_ville_pred_norm"] = out["ville_enrichie"].map(normalize_eval)
    out["_ville_ref_norm"] = out["ville_reference"].map(normalize_eval)
    out["_ville_orig_norm"] = out["Ville"].map(normalize_eval)

    out["ville_correcte"] = pd.Series(
        pd.NA,
        index=out.index,
        dtype="boolean"
    )

    out.loc[annotated_mask, "ville_correcte"] = (
        out.loc[annotated_mask, "_ville_pred_norm"]
        ==
        out.loc[annotated_mask, "_ville_ref_norm"]
    )

    # NORMALISATION DES STATUTS ET TYPES DE DÉGRADATION


    out["_ville_statut_norm"] = out["ville_statut"].map(normalize_label)
    out["_degradation_norm"] = out["ville_type_degradation"].map(normalize_label)

    healthy_labels_norm = {
        normalize_label(label)
        for label in healthy_labels
    }

    resolved_statuses = {
        "saine",
        "corrige_auto",
        "a_verifier"
    }

    out["resolved"] = (
        out["_ville_statut_norm"].isin(resolved_statuses)
        &
        ~out["ville_enrichie"].map(_is_missing_eval)
    )

    out["automatic_correction"] = (
        out["_ville_statut_norm"].eq("corrige_auto")
        &
        ~out["ville_enrichie"].map(_is_missing_eval)
    )

    out["human_verification"] = out["_ville_statut_norm"].eq("a_verifier")
    out["unresolved"] = out["_ville_statut_norm"].eq("non_resolu")

    # VALIDATION RÉFÉRENTIELLE


    if validator is None:
        raise ValueError(
            "Vous devez fournir une fonction validator. Exemple :\n"
            "validator=lambda city: "
            "referential.match_city(city) is not None"
        )

    out["valide_referentiel"] = False

    prediction_mask = ~out["ville_enrichie"].map(_is_missing_eval)

    out.loc[prediction_mask, "valide_referentiel"] = (
        out.loc[prediction_mask, "ville_enrichie"]
        .apply(lambda city: bool(validator(city)))
    )

    # SOUS-ENSEMBLES D'ÉVALUATION

    evaluated = out.loc[annotated_mask].copy()

    resolved_records = evaluated.loc[
        evaluated["resolved"]
    ].copy()

    automatic_records = evaluated.loc[
        evaluated["automatic_correction"]
    ].copy()

    predicted_records = evaluated.loc[
        ~evaluated["ville_enrichie"].map(_is_missing_eval)
    ].copy()


    degraded_mask = ~evaluated["_degradation_norm"].isin(healthy_labels_norm)

    degraded_records = evaluated.loc[
        degraded_mask
    ].copy()

    healthy_records = evaluated.loc[
        ~degraded_mask
    ].copy()

    if debug:
        print("\n=== DEBUG TYPES DE DÉGRADATION ===")
        print(
            evaluated["_degradation_norm"]
            .replace("", "<vide>")
            .value_counts(dropna=False)
        )
        print(f"Libellés sains reconnus : {sorted(healthy_labels_norm)}")
        print(f"Lignes saines annotées : {len(healthy_records)}")
        print(f"Lignes dégradées annotées : {len(degraded_records)}")
        print(
            f"Vérification : {len(healthy_records)} + "
            f"{len(degraded_records)} = {len(evaluated)}"
        )

    if len(healthy_records) + len(degraded_records) != len(evaluated):
        raise RuntimeError(
            "Erreur lors de la séparation entre lignes saines et dégradées."
        )

    # MÉTRIQUES DE QUALITÉ

    resolution_rate = (
        evaluated["resolved"].mean()
        if len(evaluated) > 0
        else np.nan
    )

    end_to_end_accuracy = (
        evaluated["ville_correcte"]
        .fillna(False)
        .astype(bool)
        .mean()
        if len(evaluated) > 0
        else np.nan
    )

    accuracy_on_resolved = (
        resolved_records["ville_correcte"]
        .fillna(False)
        .astype(bool)
        .mean()
        if len(resolved_records) > 0
        else np.nan
    )

    referential_validation_rate = (
        predicted_records["valide_referentiel"].mean()
        if len(predicted_records) > 0
        else np.nan
    )

    auto_correction_accuracy = (
        automatic_records["ville_correcte"]
        .fillna(False)
        .astype(bool)
        .mean()
        if len(automatic_records) > 0
        else np.nan
    )

    # MÉTRIQUES D'EFFICIENCE

    # Nombre moyen d'agents activés

    if "nb_agents_actives" in out.columns:
        agents_activated = pd.to_numeric(
            out["nb_agents_actives"],
            errors="coerce"
        ).fillna(0)

        total_agents_activated = agents_activated.sum()
        mean_agents_activated = agents_activated.mean()

        mean_agents_activated_degraded = (
            pd.to_numeric(
                degraded_records["nb_agents_actives"],
                errors="coerce"
            )
            .fillna(0)
            .mean()
            if len(degraded_records) > 0
            else np.nan
        )
    else:
        total_agents_activated = np.nan
        mean_agents_activated = np.nan
        mean_agents_activated_degraded = np.nan

    # Nombre d'appels LLM

    if "n_appels_llm" in out.columns:
        llm_calls = pd.to_numeric(
            out["n_appels_llm"],
            errors="coerce"
        ).fillna(0)

        total_llm_calls = llm_calls.sum()
        mean_llm_calls = llm_calls.mean()

        mean_llm_calls_degraded = (
            pd.to_numeric(
                degraded_records["n_appels_llm"],
                errors="coerce"
            )
            .fillna(0)
            .mean()
            if len(degraded_records) > 0
            else np.nan
        )
    else:
        total_llm_calls = np.nan
        mean_llm_calls = np.nan
        mean_llm_calls_degraded = np.nan

    # Latence

    if "latence_s" in out.columns:
        latency = pd.to_numeric(
            out["latence_s"],
            errors="coerce"
        )

        mean_latency = latency.mean()
        median_latency = latency.median()
        p95_latency = latency.quantile(0.95)

        mean_latency_degraded = (
            pd.to_numeric(
                degraded_records["latence_s"],
                errors="coerce"
            ).mean()
            if len(degraded_records) > 0
            else np.nan
        )
    else:
        mean_latency = np.nan
        mean_latency_degraded = np.nan
        median_latency = np.nan
        p95_latency = np.nan

    # TAUX DES DIFFÉRENTS STATUTS

    auto_correction_rate = (
        evaluated["automatic_correction"].mean()
        if len(evaluated) > 0
        else np.nan
    )

    human_verification_rate = (
        evaluated["human_verification"].mean()
        if len(evaluated) > 0
        else np.nan
    )

    unresolved_rate = (
        evaluated["unresolved"].mean()
        if len(evaluated) > 0
        else np.nan
    )

    # TABLEAU RÉCAPITULATIF

    summary = pd.DataFrame([{
        "Architecture": architecture,

        "N Total": len(out),
        "N Annotated": len(evaluated),
        "N Healthy Annotated Records": len(healthy_records),
        "N Degraded Records": len(degraded_records),
        "N Resolved": len(resolved_records),
        "N Automatic Corrections": len(automatic_records),

        "Resolution Rate": resolution_rate,
        "End-to-End Accuracy": end_to_end_accuracy,
        "Accuracy on Resolved Records": accuracy_on_resolved,
        "Referential Validation Rate": referential_validation_rate,
        "Auto-Correction Accuracy": auto_correction_accuracy,

        "Auto-Correction Rate": auto_correction_rate,
        "Human Verification Rate": human_verification_rate,
        "Unresolved Rate": unresolved_rate,

        "Total Agents Activated": total_agents_activated,
        "Mean Agents Activated per Record": mean_agents_activated,
        "Mean Agents Activated on Degraded Records": mean_agents_activated_degraded,

        "Total LLM Calls": total_llm_calls,
        "Mean LLM Calls per Record": mean_llm_calls,
        "Mean LLM Calls on Degraded Records": mean_llm_calls_degraded,

        "Mean Latency (s)": mean_latency,
        "Mean Latency on Degraded Records (s)": mean_latency_degraded,
        "Median Latency (s)": median_latency,
        "P95 Latency (s)": p95_latency
    }])

    # PERFORMANCE PAR TYPE DE DÉGRADATION

    robustness_rows = []

    for degradation_type, group in degraded_records.groupby(
        "_degradation_norm",
        dropna=False
    ):
        group_resolved = group.loc[group["resolved"]]
        group_auto = group.loc[group["automatic_correction"]]
        group_predictions = group.loc[
            ~group["ville_enrichie"].map(_is_missing_eval)
        ]

        robustness_rows.append({
            "Degradation Type": (
                degradation_type
                if degradation_type != ""
                else "<vide>"
            ),
            "N": len(group),

            "Resolution Rate": group["resolved"].mean(),

            "End-to-End Accuracy": (
                group["ville_correcte"]
                .fillna(False)
                .astype(bool)
                .mean()
            ),

            "Accuracy on Resolved Records": (
                group_resolved["ville_correcte"]
                .fillna(False)
                .astype(bool)
                .mean()
                if len(group_resolved) > 0
                else np.nan
            ),

            "Referential Validation Rate": (
                group_predictions["valide_referentiel"].mean()
                if len(group_predictions) > 0
                else np.nan
            ),

            "Auto-Correction Accuracy": (
                group_auto["ville_correcte"]
                .fillna(False)
                .astype(bool)
                .mean()
                if len(group_auto) > 0
                else np.nan
            )
        })

    robustness = pd.DataFrame(robustness_rows)

    # ANALYSE DES ERREURS

    error_columns = [
        column
        for column in [
            "Ville",
            "ville_enrichie",
            "ville_reference",
            "ville_statut",
            "ville_score",
            "ville_type_degradation",
            "_degradation_norm",
            "ville_source",
            "ville_decision_log",
            "valide_referentiel",
            "agents_actives",
            "nb_agents_actives",
            "n_appels_llm",
            "latence_s"
        ]
        if column in evaluated.columns
    ]

    errors = evaluated.loc[
        ~evaluated["ville_correcte"].fillna(False).astype(bool),
        error_columns
    ].copy()

    unannotated_columns = [
        column
        for column in [
            "Ville",
            "ville_enrichie",
            "ville_reference",
            "ville_statut",
            "ville_score",
            "ville_type_degradation",
            "_degradation_norm",
            "ville_source",
            "ville_decision_log",
            "agents_actives",
            "nb_agents_actives",
            "n_appels_llm",
            "latence_s"
        ]
        if column in out.columns
    ]

    unannotated = out.loc[
        ~annotated_mask,
        unannotated_columns
    ].copy()

    # AFFICHAGE

    print("=" * 70)
    print(f"ÉVALUATION {architecture}")
    print("=" * 70)

    try:
        from IPython.display import display
        display(summary.T)

        if not robustness.empty:
            print("\nPERFORMANCE PAR TYPE DE DÉGRADATION")
            display(robustness)

        print(
            f"\nErreurs sur les lignes annotées : "
            f"{len(errors)} / {len(evaluated)}"
        )

        if len(errors) > 0:
            display(errors.head(20))

    except ImportError:
        print(summary.T)

    print(
        f"\nLignes sans vérité terrain : "
        f"{len(unannotated)} / {len(out)}"
    )

    # 14. EXPORT EXCEL

    if export_path:
        with pd.ExcelWriter(
            export_path,
            engine="openpyxl"
        ) as writer:

            summary.to_excel(
                writer,
                sheet_name="Summary",
                index=False
            )

            if not robustness.empty:
                robustness.to_excel(
                    writer,
                    sheet_name="By_Degradation_Type",
                    index=False
                )

            errors.to_excel(
                writer,
                sheet_name="Errors",
                index=False
            )

            unannotated.to_excel(
                writer,
                sheet_name="Unannotated",
                index=False
            )

            degraded_records.to_excel(
                writer,
                sheet_name="Degraded_Records",
                index=False
            )

            healthy_records.to_excel(
                writer,
                sheet_name="Healthy_Records",
                index=False
            )

            out.to_excel(
                writer,
                sheet_name="Detailed_Results",
                index=False
            )

        print(f"Résultats exportés dans : {export_path}")

    return {
        "summary": summary,
        "robustness": robustness,
        "errors": errors,
        "unannotated": unannotated,
        "degraded_records": degraded_records,
        "healthy_records": healthy_records,
        "detailed_results": out
    }

In [ ]:
mas_evaluation = evaluate_mas_with_city_ground_truth(
    df_result=df_result,
    ground_truth_path="rcp_data_degrade_v2_solution.csv",
    architecture="Multi-Agent",
    validator=lambda city: referential.match_city(city) is not None,
    export_path="evaluation_mas_ground_truth.xlsx",
    healthy_labels=("saine",),
    debug=True
)